# Bidirectional Requirements Traceability with Reproducible Evidence

**Michael Zargham — Dynamical Systems Group**
JAXA MBSE/MCE Workshop · Presentations from Industry · 2026-06-12

---

How do we keep records of **design judgment** auditable across heterogeneous
tools, disciplines, and time?

Two ideas carry this whole demo:

1. **Evidence does not verify requirements; only human attestation does.**
   A SymPy proof or a scipy simulation produces claims true *within a model*.
   The engineer judges whether the model is *adequate* and the evidence
   *sufficient*. That judgment is the boundary.
2. **The vendor-neutral semantic spine is *how* we make provenance work
   across distinct authoritative sources of truth (ASoTs)** — code in git,
   RDF in Flexo, runtime in Docker — stitched by standard W3C vocabularies.

This notebook is the live walkthrough. Every cell shows exactly what runs.

In [1]:
import os, json
from pathlib import Path

from rdflib import Dataset, URIRef

from ontology.prefixes import (
    ADCS, RTM, EARL, PROV,
    G_AUDIT, G_EVIDENCE, G_ATTESTATIONS, G_STRUCTURAL,
    NAMED_GRAPHS,
)

# Run from the repo root so relative paths (ontology/, structural/) resolve.
print("cwd:", Path.cwd())
print("environment ready")

cwd: /Users/z/Documents/GitHub/ADCS-lifecycle-demo
environment ready


## 1 · The spine, assembled like packages

`rtm:` is a *thin integration ontology*. It imports established standards and
adds only convenience subclasses/subproperties — **no novel epistemic
vocabulary**. The build enforces a triple budget so it cannot quietly grow new
terms.

In [2]:
manifest = json.loads(Path("ontology/assembly_manifest.json").read_text())
art = manifest["artifact"]

print(f"rtm.ttl — {art['total_triples']} triples "
      f"(budget {manifest['triple_budget']['value']}, "
      f"{manifest['triple_budget']['headroom']} headroom)")
print(f"ROBOT/ELK verified at build: {manifest['robot_used']}")
print()
print("Local glue — the project's entire contribution:")
print(f"  {art['subclass_axioms']:>3} subclass axioms")
print(f"  {art['subproperty_axioms']:>3} subproperty axioms")
print(f"  {art['equivalence_axioms']:>3} equivalence axioms (SysMLv2 OWL alignment)")
print()
print("Imported upstream vocabularies (the spine):")
for name, info in sorted(manifest["imports"].items()):
    print(f"  {name:9s} {info['total_triples']:>5} triples  "
          f"{info['referenced_count']} terms used  {info['iri']}")

rtm.ttl — 241 triples (budget 450, 209 headroom)
ROBOT/ELK verified at build: False

Local glue — the project's entire contribution:
   21 subclass axioms
    9 subproperty axioms
    9 equivalence axioms (SysMLv2 OWL alignment)

Imported upstream vocabularies (the spine):
  EARL        162 triples  5 terms used  http://www.w3.org/ns/earl#
  OSLC QM      90 triples  1 terms used  http://open-services.net/ns/qm#
  OSLC RM      74 triples  1 terms used  http://open-services.net/ns/rm#
  OntoGSN    3784 triples  3 terms used  https://w3id.org/OntoGSN/ontology
  P-PLAN      154 triples  0 terms used  http://purl.org/net/p-plan
  PROV-O     1146 triples  10 terms used  http://www.w3.org/ns/prov-o


In [3]:
# Each lifecycle layer lives in its own named graph (Flexo branch-shaped).
print("Named-graph quadstore layout:")
for layer, iri in NAMED_GRAPHS.items():
    print(f"  <{layer:14s}>  {iri}")

Named-graph quadstore layout:
  <ontology      >  http://example.org/adcs-demo/graph/ontology
  <plan          >  http://example.org/adcs-demo/graph/plan
  <structural    >  http://example.org/adcs-demo/graph/structural
  <context       >  http://example.org/adcs-demo/graph/context
  <evidence      >  http://example.org/adcs-demo/graph/evidence
  <attestations  >  http://example.org/adcs-demo/graph/attestations
  <plan_execution>  http://example.org/adcs-demo/graph/plan-execution
  <audit         >  http://example.org/adcs-demo/graph/audit


## 2 · The system and its requirements  ·  *ASoT: git / RDF*

The structural model is SysMLv2 instance data in RDF: a geostationary comms
satellite (`sat:GeoSat`) and its ADCS subsystem, with four allocated
requirements. This graph is version-controlled in git — the first
authoritative source of truth.

In [4]:
from analysis.load_params import load_structural_graph, load_params

g_struct = load_structural_graph()
params = load_params(g_struct)

q = '''
PREFIX sysml: <https://www.omg.org/spec/SysML/2.0/>
SELECT ?id ?text WHERE {
  ?r a sysml:RequirementDefinition ; sysml:declaredName ?id ; sysml:text ?text .
  FILTER(STRSTARTS(?id, "REQ-"))
} ORDER BY ?id
'''
for row in g_struct.query(q):
    text = " ".join(str(row.text).split())
    print(f"{row.id}: {text}\n")

REQ-001: The ADCS shall maintain pointing accuracy within 0.1 degrees (3-sigma) under nominal disturbance conditions within 120 seconds of a slew maneuver.

REQ-002: Reaction wheel angular momentum shall not exceed the rated maximum momentum capacity (4.0 N.m.s) during nominal operations.

REQ-003: The closed-loop ADCS shall be asymptotically stable with all eigenvalues having real parts less than or equal to -0.010 rad/s.

REQ-004: The ADCS shall reject gravity gradient disturbance torques encountered in geostationary orbit without exceeding actuator torque capacity.



In [5]:
# Key structural parameters that flow into the behavior model.
for k in ("mass", "Kp", "Kd", "maxMomentum", "maxTorque", "orbitalRate"):
    if k in params:
        print(f"  {k:14s} = {params[k]}")

  mass           = 550.0
  Kp             = 1.0
  Kd             = 10.0
  maxMomentum    = 4.0
  maxTorque      = 0.1
  orbitalRate    = 7.2722e-05


## 3 · Traceable oracles for behavior models  ·  *the focus*

A **behavior model** here is the closed-loop ADCS dynamics, integrated with
scipy. A **traceable oracle** runs that model and emits a *machine-checkable,
content-addressed, provenance-linked* outcome: it compares a model-output
metric against the requirement's acceptance criterion.

Crucial distinction: the oracle performs **verification** — a fully-specified
comparison of a **model-level claim** ("settling time exceeds the 120 s
budget, *within the model*"). It is `earl:mode = earl:automatic`. It does **not**
assert that the physical requirement is satisfied. Only human attestation does.

In [6]:
from analysis.numerical import run_step_response
from analysis.symbolic import run_symbolic_analysis

# Run the numerical oracle's behavior model: a 10-degree slew, default gains.
sim = run_step_response(params)
summary = sim.summary()
print("Behavior-model output (scipy):")
for k in ("settling_time_s", "peak_wheel_momentum", "peak_error_deg"):
    print(f"  {k:22s} = {summary[k]:.3f}")

# Symbolic oracle supplies REQ-003's metric (dominant eigenvalue real part).
sym = run_symbolic_analysis(params)
worst_real_part = max(sym.stability_margins.values())
print(f"  {'worst_real_part':22s} = {worst_real_part:.4f}  (symbolic)")

Behavior-model output (scipy):
  settling_time_s        = 310.626
  peak_wheel_momentum    = 0.810
  peak_error_deg         = 9.987


  worst_real_part        = -0.0153  (symbolic)


In [7]:
from analysis.oracle import evaluate_requirement_oracle, ACCEPTANCE_CRITERIA

# One merged metric dict the oracle reads from (numerical + symbolic).
metrics = dict(summary)
metrics["worst_real_part"] = worst_real_part

print("Traceable-oracle outcomes (model-level VERIFICATION):\n")
oracle_results = {}
for req in ("REQ-001", "REQ-002", "REQ-003", "REQ-004"):
    r = evaluate_requirement_oracle(metrics, req)
    oracle_results[req] = r
    print(f"  {req}  ->  earl:{r.outcome:9s}  {r.detail}")

print()
print("REQ-004 has no machine-readable scalar budget -> the oracle returns")
print("cantTell rather than fabricating a verdict it cannot compute.")

Traceable-oracle outcomes (model-level VERIFICATION):

  REQ-001  ->  earl:failed     settling_time_s = 310.626 le 120 s -> fail (model-level)
  REQ-002  ->  earl:passed     peak_wheel_momentum = 0.809768 le 4 N.m.s -> pass (model-level)
  REQ-003  ->  earl:passed     worst_real_part = -0.0152841 le -0.01 rad/s -> pass (model-level)
  REQ-004  ->  earl:cantTell   no machine-readable acceptance criterion for this requirement

REQ-004 has no machine-readable scalar budget -> the oracle returns
cantTell rather than fabricating a verdict it cannot compute.


In [8]:
from traceability.oracle_assertion import emit_oracle_assertion

# Persist the REQ-001 oracle outcome as an earl:Assertion and show the RDF.
demo = Dataset()
ev = URIRef("urn:adcs:evidence:EV-SIM-REQ-001")
emit_oracle_assertion(demo, ev, ADCS["REQ-001"], oracle_results["REQ-001"])

print(demo.graph(URIRef(G_AUDIT)).serialize(format="turtle"))

@prefix ns1: <http://www.w3.org/ns/earl#> .
@prefix ns2: <http://example.org/ontology/rtm#> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<urn:adcs:assertion:oracle-REQ-001-2026-06-11T06-55-42-742470-00-00> a ns2:BehaviorOracleAssertion,
        ns1:Assertion,
        prov:Activity ;
    ns2:evaluatesAgainst <http://example.org/adcs-demo/REQ-001> ;
    ns2:metricKey "settling_time_s" ;
    ns2:metricValue 310.62618149585654 ;
    ns2:thresholdValue 120.0 ;
    ns1:mode ns1:automatic ;
    ns1:outcome ns1:failed ;
    ns1:subject <urn:adcs:evidence:EV-SIM-REQ-001> ;
    ns1:test <urn:adcs:test:behavior-oracle> ;
    prov:atTime "2026-06-11T06:55:42.742470+00:00"^^xsd:dateTime ;
    prov:wasAssociatedWith <urn:adcs:agent:behavior-oracle> .




Read the triples: `earl:subject` is the **evidence** under test; the
requirement is linked by `rtm:evaluatesAgainst` (subPropertyOf `prov:used`) —
**never** `rtm:attests`, which is reserved for the human. The oracle records
*what it computed*, not a judgment of satisfaction.

## 4 · The full lifecycle, then attestation = the human boundary

Now run the whole tested lifecycle locally. Watch the stage banners: assemble
ontology → load structural → symbolic → numerical → bind evidence → attest →
closure-rule verification → audit. It returns the populated `Dataset`.

In [9]:
from pipeline.runner import run_pipeline

ds = run_pipeline(auto_attest=True, backend="local")
print("\nlifecycle complete — Dataset populated")


[Preflight] Probing backends...
  Compute: Local in-process compute (host=Michaels-MacBook-Pro-2.local, python=3.12.12)
  Storage: Local filesystem (rdflib Dataset, persisted as output/rtm.ttl + output/rtm.trig)
  Compute probe: PASS
  Storage probe: PASS
  Operating org: urn:adcs:org:local-operator (Local Operator)

[Stage 0/8] Ontology Assembly
──────────────────────────────────────────────────────────────────────────────
  Loading assembled rtm.ttl (built 2026-05-29T03:22:19Z from rtm-edit.ttl)
  Imports resolved:
    OK  EARL         162 triples,  5 terms referenced
    OK  OSLC QM       90 triples,  1 terms referenced
    OK  OSLC RM       74 triples,  1 terms referenced
    OK  OntoGSN     3784 triples,  3 terms referenced
    OK  P-PLAN       154 triples,  0 terms referenced
    OK  PROV-O      1146 triples,  10 terms referenced
  SysMLv2 equivalence axioms: 9
  Local rtm: integration glue: 21 subclass + 9 subproperty axioms
  Verification: Python assembly only (no-Java path; r

  Inertia: Jxx=327.1, Jyy=107.5, Jzz=303.4 kg.m^2
  Stability margins: x=-0.0153, y=-0.0465, z=-0.0165
  Proof REQ-001: VERIFIED (3 lemmas)
  Proof REQ-002: VERIFIED (3 lemmas)
  Proof REQ-003: VERIFIED (4 lemmas)
  Proof REQ-004: VERIFIED (3 lemmas)

[Stage 3] Running numerical simulations...
  Step response: settling=310.6s, final_error=0.0072 deg
  Peak wheel momentum: 0.810 N.m.s (limit: 4.0)
  Peak control torque: 0.0436 N.m (limit: 0.1)


  Disturbance rejection: peak_error=0.001000 deg

[Stage 4] Binding evidence to RDF graph...
  Evidence artifacts created: 135 nodes (written to <adcs:evidence>)

[Stage 5] Assembling RTM...
  Evidence completeness: PASS (all requirements have evidence)
  Pre-attestation RTM exported to output/rtm_pre_attestation.{ttl,trig}

[Stage 6] Human attestation...

  REQ-001: ATTESTATION DECLINED
    Settling time 311s exceeds 120s requirement.
    Action: retune gains (Kp: 1->4, Kd: 10->30) and re-verify.

  Requirement: REQ-001
  The ADCS shall maintain pointing accuracy within 0.1 degrees
        (3-sigma) under nominal disturbance conditions within 120 seconds
        of a slew maneuver.
  Derived from: SAT-REQ-POINTING
  Allocated to: PDController, StarTracker

  Evidence (2 artifacts):
    [ProofArtifact] hash=a7f1bfb5ca74cfae...
      Symbolic proof: Steady-state pointing error under constant disturbance is 2*tau_gg/Kp, bounded by actuator/gain design
    [SimulationResult] hash=068c61db

  Closure-rule verification: PASS
    SHACL shapes:        0 violations
    Re-verification:     0 mismatches
  Closure-rule assertion: urn:adcs:assertion:closure-2026-06-11T06-55-44-054432-00-00

[Stage 7a] Auditing forward / backward / bidirectional traceability...
  Forward    PASS  (4 checked, 0 failures)
  Backward   PASS  (4 checked, 0 failures)
  Bidirectional: PASS
  Orphans: none
  Audit report -> output/audit.md, output/audit.csv

[Stage 7] Generating reports...
  Backend: Local filesystem (rdflib Dataset, persisted as output/rtm.ttl + output/rtm.trig)
  Persisted 8 named graphs (1107 triples total)
REQUIREMENTS TRACEABILITY MATRIX — STATUS SUMMARY

  REQ-001: The ADCS shall maintain pointing accuracy within 0.1 degrees...
    Status: ATTESTED (earl:failed) — engineering finding

  REQ-002: Reaction wheel angular momentum shall not exceed the rated  ...
    Status: ATTESTED (earl:passed)

  REQ-003: The closed-loop ADCS shall be asymptotically stable with    ...
    Status: A

In [10]:
# Evidence binding (Stage 4): every artifact is content-addressed and
# addresses a requirement. (rtm:addresses = structural intent, NOT a verdict.)
q = '''
PREFIX rtm: <http://example.org/ontology/rtm#>
SELECT ?ev ?req ?hash ?summary WHERE {
  ?ev a rtm:SimulationResult ;
      rtm:addresses ?req ;
      rtm:contentHash ?hash ;
      rtm:resultSummary ?summary .
} ORDER BY ?req
'''
for row in ds.query(q):
    print(f"{str(row.req).split('/')[-1]}  hash={str(row.hash)[:16]}...")
    print(f"   {row.summary}\n")

REQ-001  hash=068c61dbfae35f13...
   Step response: settling=310.6s, final_error=0.0072 deg

REQ-002  hash=068c61dbfae35f13...
   Peak wheel momentum: 0.810 N.m.s (limit=4.0)

REQ-004  hash=d19e0be5468eddb0...
   Disturbance rejection: peak_error=0.001000 deg



In [11]:
# Attestation (Stage 6): REQ-001 is attested earl:failed — recorded, not hidden.
# Each attestation carries adequacy (gsn:Assumption) + sufficiency
# (gsn:Justification): the Hawkins-Habli Assurance Claim Point split.
q = '''
PREFIX rtm: <http://example.org/ontology/rtm#>
PREFIX gsn: <https://w3id.org/OntoGSN/ontology#>
SELECT ?req ?outcome ?stmt WHERE {
  ?a a rtm:Attestation ; rtm:attests ?req ; rtm:hasOutcome ?outcome ;
     gsn:inContextOf ?ctx . ?ctx gsn:statement ?stmt .
} ORDER BY ?req
'''
for row in ds.query(q):
    req = str(row.req).split("/")[-1]
    outcome = str(row.outcome).split("#")[-1]
    print(f"{req}  earl:{outcome}\n   - {row.stmt}\n")

REQ-001  earl:failed
   - Step-response simulation is adequate for evaluating pointing-accuracy settling time at this point in the lifecycle.

REQ-001  earl:failed
   - Evidence is sufficient to conclude REQ-001 is NOT yet satisfied: settling time 311s exceeds the 120s requirement. Action item: retune gains (Kp: 1->4, Kd: 10->30) and re-verify.

REQ-002  earl:passed
   - Energy-based momentum bound is conservative. Reaction wheel model adequate for peak momentum estimation.

REQ-002  earl:passed
   - Symbolic bound and simulation both confirm peak momentum well below 4.0 N.m.s.

REQ-003  earl:passed
   - Linearized stability analysis via Routh-Hurwitz is adequate. Nonlinear effects are second-order for small angles.

REQ-003  earl:passed
   - Routh-Hurwitz proof confirms asymptotic stability for all positive J, Kp, Kd. Numerical eigenvalues confirm margins exceed -0.010 rad/s on all axes.

REQ-004  earl:passed
   - Linearized gravity gradient model adequate for GEO orbit. Higher-order 

## 5 · Verification vs validation, side by side

Fold the oracle outcomes into the lifecycle Dataset's audit graph. The audit
graph now holds **three families of automatic verification** (closure-rule,
behavior-oracle, and — under `--compute=docker` — digest-match), each an
`earl:Assertion` with `earl:mode = earl:automatic`, sitting **beside** the
human attestations. Verification is system-internal; attestation is the human
boundary.

In [12]:
def evidence_for(ds, req):
    q = f'''
    PREFIX rtm: <http://example.org/ontology/rtm#>
    SELECT ?e WHERE {{ ?e rtm:addresses <{ADCS[req]}> }} LIMIT 1
    '''
    rows = list(ds.query(q))
    return rows[0].e if rows else URIRef(f"urn:adcs:evidence:{req}")

for req, r in oracle_results.items():
    emit_oracle_assertion(ds, evidence_for(ds, req), ADCS[req], r)

# Tally what now lives in the audit graph by EARL mode.
q = '''
PREFIX earl: <http://www.w3.org/ns/earl#>
PREFIX rtm: <http://example.org/ontology/rtm#>
SELECT ?mode (COUNT(?a) AS ?n) WHERE {
  ?a a earl:Assertion ; earl:mode ?mode .
} GROUP BY ?mode
'''
print("EARL assertions in the Dataset by mode:")
for row in ds.query(q):
    print(f"  {str(row.mode).split('#')[-1]:10s}  {row.n}")

EARL assertions in the Dataset by mode:
  automatic   5


## 6 · Provenance across distinct authoritative sources of truth  ·  *the point*

The same `Dataset` persists identically to local disk, **Flexo MMS**, or bare
Fuseki. Here we push it to the **Planetary Utilities Flexo instance** live, then
read it back to prove it landed. Code is in git; RDF is in Flexo; runtime is in
Docker — three ASoTs, stitched by standard PROV edges + `rtm:gitRef` +
`rtm:flexoRecord`.

A fail-fast `probe()` runs first. If `FLEXO_TOKEN` is unset (offline rehearsal),
we skip the remote and still demonstrate the trust queries against the local
Dataset — the provenance chain is identical.

In [13]:
import httpx
from pipeline.backends.flexo import FlexoBackend
from pipeline.backends.base import BackendUnavailable

flexo = FlexoBackend()
flexo_live = False

# A live push needs a credential, not just a reachable host (the HEAD probe
# is unauthenticated). Gate on FLEXO_TOKEN, then probe for reachability.
if not flexo.token:
    print("FLEXO_TOKEN not set — offline rehearsal mode.")
    print("Trust queries below run identically against the local Dataset.")
else:
    try:
        flexo.probe()
        flexo_live = True
        print(f"Flexo reachable: {flexo.describe()}")
    except BackendUnavailable as exc:
        print(f"Flexo unreachable ({exc}). Continuing with local trust queries.")

FLEXO_TOKEN not set — offline rehearsal mode.
Trust queries below run identically against the local Dataset.


In [14]:
if flexo_live:
    try:
        counts = flexo.persist(ds, Path("output"))
        print("pushed named graphs to Flexo:")
        for graph_iri, n in counts.items():
            print(f"  {n:>5} triples  ->  {graph_iri}")
        print(f"\nstable record URI: {flexo.record_uri('attestations')}")
    except (BackendUnavailable, httpx.HTTPError) as exc:
        flexo_live = False
        print(f"live push failed ({exc}); falling back to local trust queries.")
else:
    print("(skipped live push — offline rehearsal)")

(skipped live push — offline rehearsal)


In [15]:
import httpx

if flexo_live:
    # Read the attestation back from Flexo to prove it round-tripped.
    sparql = '''
    PREFIX rtm: <http://example.org/ontology/rtm#>
    PREFIX adcs: <http://example.org/adcs-demo/>
    ASK { adcs:ATT-REQ-003 a rtm:Attestation }
    '''
    url = (f"{flexo.url}/orgs/{flexo.org}/repos/{flexo.repo}"
           f"/branches/{flexo.branch_prefix}attestations/query")
    r = httpx.post(url, headers={
        "Authorization": f"Bearer {flexo.token}",
        "Content-Type": "application/sparql-query",
        "Accept": "application/json",
    }, content=sparql, timeout=30.0)
    print(f"read-back ASK -> {r.json()}")
else:
    print("(skipped live read-back — no FLEXO_TOKEN)")

(skipped live read-back — no FLEXO_TOKEN)


In [16]:
# The trust queries answer "how can I trust this?" — backend-agnostic, they
# walk evidence -> activity -> container -> image -> git ref + auspices.
from traceability.queries import trust_summary, render_trust_summary

ev_iri = str(ADCS["EV-SIM-REQ-001"])
print(render_trust_summary(trust_summary(ds, ev_iri)))

Trust panel for http://example.org/adcs-demo/EV-SIM-REQ-001
Technical provenance:
  activity:    http://example.org/adcs-demo/NS-REQ-001
  container:   - (id=-)
  image:       http://example.org/adcs-demo/EV-PROOF-REQ-001
  digest:      a7f1bfb5ca74cfae7e8891b5caecb67eab6104b1eb0f838ff14c13ee683b829f
  git ref:     -
  host:        urn:adcs:location:local:Michaels-MacBook-Pro-2.local
  executor:    http://example.org/adcs-demo/ScipyEngine
Auspices:
  operating:   -
  hosting:     Local Operator
Closure-rule assertions: 1
  - outcome=http://www.w3.org/ns/earl#passed violations=0


## 7 · Close the loop  ·  *push the button*

REQ-001 failed. The engineer retunes the controller gains. The model changes,
so its hash changes — **every prior proof and simulation bound to the old model
is invalidated**. Re-run the behavior model and the oracle: settling time drops
below the 120 s budget, and the oracle outcome flips to `passed`. The loop is
auditable because every step is traceable across the ASoTs — and the
vendor-neutral spine is what made that possible.

In [17]:
retuned = dict(params)
retuned["Kp"], retuned["Kd"] = 4.0, 30.0   # was 1.0 / 10.0

sim2 = run_step_response(retuned)
s2 = sim2.summary()
r2 = evaluate_requirement_oracle({**s2, "worst_real_part": worst_real_part}, "REQ-001")

print(f"REQ-001 settling time:  {summary['settling_time_s']:.0f} s  (Kp=1,  Kd=10)")
print(f"                  ->    {s2['settling_time_s']:.0f} s  (Kp=4,  Kd=30)")
print(f"\noracle re-verification:  earl:{r2.outcome}  —  {r2.detail}")
print("\nThe engineer would now re-attest REQ-001 against the v2 model.")
print("Evidence, oracle outcomes, and attestations all trace across git +")
print("Flexo + Docker. That is reproducible evidence with full provenance.")

REQ-001 settling time:  311 s  (Kp=1,  Kd=10)
                  ->    78 s  (Kp=4,  Kd=30)

oracle re-verification:  earl:passed  —  settling_time_s = 77.9662 le 120 s -> pass (model-level)

The engineer would now re-attest REQ-001 against the v2 model.
Evidence, oracle outcomes, and attestations all trace across git +
Flexo + Docker. That is reproducible evidence with full provenance.
